In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.amp import autocast, GradScaler

from peft import LoraConfig, get_peft_model
from tqdm import tqdm

from Modules.datasets import BlipRAGDataset, BlipDataCollator
from Modules.embedding_module import Embedder
from Modules.retrieval_module import Retriever

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [3]:
TRAIN_IMAGE_DIR = "dataset/flickr30k_images/train"
TRAIN_CAPTIONS_DIR = "dataset/captions-train.csv"

TEST_IMAGE_DIR = "dataset/flickr30k_images/test"
TEST_CAPTIONS_DIR = "dataset/captions-test.csv"

FAISS_PATH = "flickr30k_clip_images.faiss"
TRAIN_METADATA_PATH = "train_metadata.json"
TEST_METADATA_PATH = "test_metadata.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
BLIP_model_name = "Salesforce/blip-image-captioning-base"  # or use '-large' for bigger model

# BLIP_processor = BlipProcessor.from_pretrained(BLIP_model_name, local_files_only=True, use_fast=True)
# BLIP_model = BlipForConditionalGeneration.from_pretrained(BLIP_model_name, dtype=torch.float16, local_files_only=True).to(DEVICE)
BLIP_processor = BlipProcessor.from_pretrained(BLIP_model_name, use_fast=True)
BLIP_model = BlipForConditionalGeneration.from_pretrained(BLIP_model_name, dtype=torch.float16).to(DEVICE)

In [5]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)
collator = BlipDataCollator(BLIP_processor, 128)

In [6]:
train_dataset = BlipRAGDataset(image_dir=TRAIN_IMAGE_DIR, metadata_path=TRAIN_METADATA_PATH, retriever=retriever)
test_dataset = BlipRAGDataset(image_dir=TEST_IMAGE_DIR, metadata_path=TEST_METADATA_PATH, retriever=retriever)

In [7]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [21]:
train_dataset[100]

{'image': <PIL.Image.Image image mode=RGB size=500x375>,
 'prompt': 'Similar images are described as:\n- A little boy who appears to be about 5 years old is shown kicking a white volleyball .\n- Kid in red sweatshirt ice skating\n- A soccer player in midair as he kicks the soccer ball . \n\n Describe the image.',
 'target_caption': 'A young male kneeling in front of a hockey goal with a hockey stick in his right hand .'}

In [11]:
for x in train_loader:
    print(x.keys())
    for key in x.keys():
        print(x[key].shape)
    break

dict_keys(['pixel_values', 'input_ids', 'attention_mask', 'labels'])
torch.Size([16, 3, 384, 384])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])


In [ ]:
for name, module in BLIP_model.text_decoder.named_modules():
    print(name)

In [12]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "attention.self.query",
        "attention.self.value",
        "crossattention.self.query",
        "crossattention.self.value",
    ],  # text decoder only
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

BLIP_model = BLIP_model
BLIP_model.text_decoder = get_peft_model(BLIP_model.text_decoder, lora_config)
BLIP_model.text_decoder.print_trainable_parameters() 

trainable params: 2,359,296 || all params: 140,240,444 || trainable%: 1.6823


In [13]:
for param in BLIP_model.vision_model.parameters():
    param.requires_grad = False

In [15]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, BLIP_model.parameters()),
    # lr=5e-5,
    lr=1e-4,
    weight_decay=0.01
)

scaler = GradScaler()
BLIP_model.train()

num_epochs = 10

In [16]:
for epoch in range(num_epochs):
    batch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=True)

    for batch in progress_bar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()

        with autocast(DEVICE):
            outputs = BLIP_model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    epoch_loss = batch_loss / len(train_loader)
    progress_bar.set_postfix(final_loss=epoch_loss)
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss:.4f}")

Epoch 0: 100%|██████████| 1924/1924 [12:07<00:00,  2.64it/s, loss=5.22]


Epoch 0 finished. Avg Loss: 5.2660


Epoch 1: 100%|██████████| 1924/1924 [11:58<00:00,  2.68it/s, loss=5.11]


Epoch 1 finished. Avg Loss: 5.1365


Epoch 2:   1%|          | 23/1924 [00:08<11:56,  2.65it/s, loss=5.06]


KeyboardInterrupt: 

In [43]:
for x in test_loader:
    output_ids = BLIP_model.generate(pixel_values=x['pixel_values'][0].unsqueeze(0).to(DEVICE),
                                     input_ids=x['input_ids'][0].unsqueeze(0).to(DEVICE),
                                     attention_mask=x['attention_mask'][0].unsqueeze(0).to(DEVICE),
                                     max_new_tokens=128,
                                    repetition_penalty=1.2,
                                    no_repeat_ngram_size=3,
                                    temperature=0.8,
                                    top_p=0.9,
                                    do_sample=True,)
    caption = BLIP_processor.decode(output_ids[0], skip_special_tokens=True)
    
    label = x['labels'][0]
    label[label == -100] = 0
    label =  BLIP_processor.decode(label, skip_special_tokens=True)
    print("Caption:", caption)
    print('label: ', label)
    break

Caption: similar images are described as : - three people are walking in a canyon. - a group of people are crossing a river - one woman sits on a outcropping of rock while one takes a picture and the third looks down. describe the image. looking is the red
label:  a small group of adults and children are standing on a dirt trail on a hill.


In [ ]:
BLIP_model.save_pretrained("./blip-lora")

In [ ]:
# from transformers import Trainer, TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./blip-lora",
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,
#     learning_rate=5e-5,
#     num_train_epochs=1,
#     fp16=True,
#     logging_steps=50,
#     save_steps=1000,
#     save_total_limit=2,
#     report_to="none",
#     remove_unused_columns=False,  # REQUIRED
# )

# trainer = Trainer(
#     model=BLIP_model,
#     args=training_args,
#     train_dataset=train_dataset,
#     # data_collator=BLIPDataCollator(processor),
#     data_collator=collator,
# )

# trainer.train()